# Titanic Survival Prediction Using PyTorch

## Title
**Titanic Passenger Survival Rate Analysis and Prediction Using a PyTorch Neural Network**

## Objectives
- To upload and load the Titanic dataset in Google Colab.
- To inspect the dataset, missing values and passenger characteristics.
- To calculate the overall survival rate.
- To compare survival rates by gender and passenger class.
- To detect outliers in the `Age` and `Fare` columns using the IQR method.
- To preprocess numerical and categorical features without data leakage.
- To divide the data into training, validation and testing sets.
- To build and train a feed-forward neural network using PyTorch.
- To plot separate training-loss and validation-loss graphs.
- To calculate validation and test accuracy.
- To evaluate predictions using precision, recall, F1-score and a confusion matrix.

## Theory
The Titanic dataset is used for supervised binary classification because the target variable, `Survived`, has two possible outcomes: `1` for survived and `0` for did not survive. Exploratory data analysis is used to calculate survival rates and examine relationships between survival and passenger characteristics such as gender and ticket class. Missing numerical values are replaced using the median, while missing categorical values are replaced using the most frequent category. Categorical variables are converted into numerical one-hot encoded features, and numerical features are standardised so that they have comparable scales. Outliers are detected using the Interquartile Range method, in which values below `Q1 − 1.5 × IQR` or above `Q3 + 1.5 × IQR` are flagged as unusual. A feed-forward neural network is implemented in PyTorch and trained using binary cross-entropy loss with logits. The training loss measures error on the data used to update the model, whereas validation loss measures error on unseen validation data and helps identify overfitting. Final performance is measured on a separate test set using accuracy and other classification metrics.


In [ ]:
# 1. Install required libraries
# Google Colab normally includes these packages, but this ensures availability.
!pip install -q torch pandas scikit-learn matplotlib


In [ ]:
# 2. Upload titanic.csv in Google Colab
from google.colab import files

uploaded = files.upload()
csv_path = next(iter(uploaded))
print("Loaded:", csv_path)


In [ ]:
# 3. Import libraries and set reproducibility
import warnings
warnings.filterwarnings("ignore")

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Force CPU execution as required.
device = torch.device("cpu")
print("PyTorch version:", torch.__version__)
print("Using device:", device)


In [ ]:
# 4. Load the dataset
data = pd.read_csv(csv_path)

print("Dataset shape:", data.shape)
display(data.head())


## Dataset inspection

In [ ]:
print("Dataset information:")
data.info()

print("\nMissing values:")
display(
    data.isnull()
        .sum()
        .sort_values(ascending=False)
        .to_frame("Missing Values")
)


## Survival-rate analysis

In [ ]:
total_passengers = len(data)
survivors = int(data["Survived"].sum())
non_survivors = total_passengers - survivors
survival_rate = data["Survived"].mean() * 100

print(f"Total passengers: {total_passengers}")
print(f"Survivors: {survivors}")
print(f"Non-survivors: {non_survivors}")
print(f"Overall survival rate: {survival_rate:.2f}%")


In [ ]:
survival_by_gender = (
    data.groupby("Sex")["Survived"]
        .agg(Passengers="count", Survivors="sum", Survival_Rate="mean")
)
survival_by_gender["Survival_Rate"] *= 100

display(survival_by_gender.round(2))

survival_by_gender["Survival_Rate"].plot(kind="bar")
plt.title("Survival Rate by Gender")
plt.xlabel("Gender")
plt.ylabel("Survival Rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
survival_by_class = (
    data.groupby("Pclass")["Survived"]
        .agg(Passengers="count", Survivors="sum", Survival_Rate="mean")
)
survival_by_class["Survival_Rate"] *= 100

display(survival_by_class.round(2))

survival_by_class["Survival_Rate"].plot(kind="bar")
plt.title("Survival Rate by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Survival Rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Outlier detection

Outliers in `Age` and `Fare` are detected using the IQR rule. They are displayed but not automatically deleted because unusually high fares and extreme ages can still represent genuine passengers.


In [ ]:
def find_iqr_outliers(series):
    clean_series = series.dropna()
    q1 = clean_series.quantile(0.25)
    q3 = clean_series.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_mask = (series < lower_bound) | (series > upper_bound)

    summary = {
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": int(outlier_mask.sum()),
        "Outlier Percentage": outlier_mask.mean() * 100
    }

    return summary, outlier_mask


outlier_summaries = []
outlier_masks = {}

for column in ["Age", "Fare"]:
    summary, mask = find_iqr_outliers(data[column])
    summary["Column"] = column
    outlier_summaries.append(summary)
    outlier_masks[column] = mask

outlier_summary = pd.DataFrame(outlier_summaries).set_index("Column")
display(outlier_summary.round(2))


In [ ]:
age_outliers = data.loc[
    outlier_masks["Age"],
    ["PassengerId", "Name", "Age", "Survived"]
]

fare_outliers = data.loc[
    outlier_masks["Fare"],
    ["PassengerId", "Name", "Pclass", "Fare", "Survived"]
]

print("Age outliers:")
display(age_outliers)

print("Top 20 fare outliers:")
display(fare_outliers.sort_values("Fare", ascending=False).head(20))


In [ ]:
plt.figure(figsize=(8, 4))
plt.boxplot(data["Age"].dropna(), vert=False)
plt.title("Age Distribution and Outliers")
plt.xlabel("Age")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.boxplot(data["Fare"].dropna(), vert=False)
plt.title("Fare Distribution and Outliers")
plt.xlabel("Fare")
plt.tight_layout()
plt.show()


## Data preparation

The dataset is split before fitting the preprocessing pipeline. This prevents information from the validation and test sets from influencing median values, category selection or scaling parameters.


In [ ]:
# 5. Select features and split into train, validation and test sets
feature_columns = [
    "Pclass", "Sex", "Age", "SibSp",
    "Parch", "Fare", "Embarked"
]
target_column = "Survived"

X = data[feature_columns].copy()
y = data[target_column].astype(np.float32)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print(f"Training records:   {len(X_train)}")
print(f"Validation records: {len(X_val)}")
print(f"Testing records:    {len(X_test)}")


In [ ]:
# 6. Build and fit the preprocessing pipeline
numeric_features = ["Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Pclass", "Sex", "Embarked"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Fit only on training data.
X_train_processed = preprocessor.fit_transform(X_train).astype(np.float32)
X_val_processed = preprocessor.transform(X_val).astype(np.float32)
X_test_processed = preprocessor.transform(X_test).astype(np.float32)

print("Processed training shape:", X_train_processed.shape)


In [ ]:
# 7. Create PyTorch Dataset and DataLoader objects
class TabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(
            np.asarray(labels),
            dtype=torch.float32
        ).view(-1, 1)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


batch_size = 32

train_loader = DataLoader(
    TabularDataset(X_train_processed, y_train),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    TabularDataset(X_val_processed, y_val),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    TabularDataset(X_test_processed, y_test),
    batch_size=batch_size,
    shuffle=False
)

print("DataLoaders created successfully.")


## PyTorch neural-network model

In [ ]:
# 8. Define the neural network
class SurvivalNet(nn.Module):
    def __init__(self, input_dimension):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dimension, 32),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, inputs):
        # Raw logits are returned because BCEWithLogitsLoss is used.
        return self.network(inputs)


model = SurvivalNet(
    input_dimension=X_train_processed.shape[1]
).to(device)

print(model)


In [ ]:
# 9. Define evaluation and training functions
def evaluate_model(model, data_loader, criterion):
    model.eval()

    total_loss = 0.0
    all_probabilities = []
    all_labels = []

    with torch.no_grad():
        for batch_features, batch_labels in data_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            logits = model(batch_features)
            loss = criterion(logits, batch_labels)
            probabilities = torch.sigmoid(logits)

            total_loss += loss.item() * batch_features.size(0)
            all_probabilities.extend(
                probabilities.cpu().numpy().flatten()
            )
            all_labels.extend(
                batch_labels.cpu().numpy().flatten()
            )

    average_loss = total_loss / len(data_loader.dataset)
    predictions = (np.array(all_probabilities) >= 0.5).astype(int)
    labels = np.array(all_labels).astype(int)
    accuracy = accuracy_score(labels, predictions)

    return average_loss, accuracy, predictions, np.array(all_probabilities)


def train_model(model, train_loader, val_loader, epochs=100, learning_rate=0.001):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_accuracy": [],
        "val_accuracy": []
    }

    for epoch in range(1, epochs + 1):
        model.train()

        running_loss = 0.0
        training_probabilities = []
        training_labels = []

        for batch_features, batch_labels in train_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            optimizer.zero_grad()

            logits = model(batch_features)
            loss = criterion(logits, batch_labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * batch_features.size(0)

            probabilities = torch.sigmoid(logits)
            training_probabilities.extend(
                probabilities.detach().cpu().numpy().flatten()
            )
            training_labels.extend(
                batch_labels.cpu().numpy().flatten()
            )

        train_loss = running_loss / len(train_loader.dataset)
        train_predictions = (
            np.array(training_probabilities) >= 0.5
        ).astype(int)
        train_accuracy = accuracy_score(
            np.array(training_labels).astype(int),
            train_predictions
        )

        val_loss, val_accuracy, _, _ = evaluate_model(
            model,
            val_loader,
            criterion
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_accuracy"].append(val_accuracy)

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Epoch {epoch:3d}/{epochs} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Validation Loss: {val_loss:.4f} | "
                f"Train Accuracy: {train_accuracy:.4f} | "
                f"Validation Accuracy: {val_accuracy:.4f}"
            )

    return model, history, criterion


## Model training

In [ ]:
# 10. Train the model
model, history, criterion = train_model(
    model,
    train_loader,
    val_loader,
    epochs=100,
    learning_rate=0.001
)


## Training-loss graph

The training-loss graph shows how the model's prediction error on the training set changes across epochs.


In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(9, 5))
plt.plot(epochs_range, history["train_loss"])
plt.title("Training Loss Across Epochs")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Validation-loss graph

The validation-loss graph shows error on data that was not used to update model weights. If validation loss starts increasing while training loss continues decreasing, the model may be overfitting.


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(epochs_range, history["val_loss"])
plt.title("Validation Loss Across Epochs")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
best_epoch = int(np.argmin(history["val_loss"]) + 1)
lowest_validation_loss = float(np.min(history["val_loss"]))

print(f"Lowest validation loss: {lowest_validation_loss:.4f}")
print(f"Best epoch based on validation loss: {best_epoch}")


## Accuracy and final evaluation

In [ ]:
# 11. Evaluate the final model
val_loss, val_accuracy, val_predictions, val_probabilities = evaluate_model(
    model,
    val_loader,
    criterion
)

test_loss, test_accuracy, test_predictions, test_probabilities = evaluate_model(
    model,
    test_loader,
    criterion
)

print("===== Final Results =====")
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")
print(f"Test Loss:           {test_loss:.4f}")
print(f"Test Accuracy:       {test_accuracy * 100:.2f}%")


In [ ]:
print("Classification Report:")
print(
    classification_report(
        y_test.astype(int),
        test_predictions,
        target_names=["Did Not Survive", "Survived"]
    )
)


In [ ]:
confusion = confusion_matrix(
    y_test.astype(int),
    test_predictions
)

display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=["Did Not Survive", "Survived"]
)

display_matrix.plot()
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
prediction_results = X_test.copy()
prediction_results["Actual_Survival"] = y_test.astype(int)
prediction_results["Predicted_Survival"] = test_predictions
prediction_results["Survival_Probability"] = test_probabilities

display(prediction_results.head(15))


## Discussion

The analysis shows that only about 38% of the passengers in the dataset survived, which means that non-survivors formed the larger class. Survival was strongly associated with gender and passenger class, as female passengers and first-class passengers had noticeably higher survival rates. The IQR method identified unusual values in age and fare, especially among passengers who paid very high fares. These records were retained because they can represent genuine passenger circumstances rather than data errors. The PyTorch neural network used passenger class, gender, age, family counts, fare and embarkation port to predict survival. Validation performance was measured during training, while the independent test set was used for final evaluation. The training and validation loss curves make it possible to observe whether the model learned consistently or began to overfit. Accuracy shows the overall proportion of correct predictions, while precision, recall, F1-score and the confusion matrix provide a more detailed explanation of performance for survivors and non-survivors.

## Conclusion

This experiment successfully merged exploratory Titanic analysis with a complete PyTorch binary-classification workflow that can run in Google Colab on a CPU. The notebook calculates survival rates, studies survival differences, identifies outliers, handles missing data, encodes categorical variables, standardises numerical features and trains a neural network using separate training, validation and testing data. It also produces separate training-loss and validation-loss graphs and evaluates the final model using accuracy and classification metrics. The results demonstrate that passenger information can be used to predict survival with useful accuracy, although the predictions remain statistical estimates and do not explain every individual outcome.
